# Deep Image Prior for Thermal Image Restoration
### Zero-shot Inpainting and Denoising on Single Thermal Infrared Frames

This notebook demonstrates zero-shot unsupervised image restoration using **Deep Image Prior (DIP)** with a 5-level U-Net generator.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Add ml module to path
sys.path.insert(0, os.path.abspath('..'))

from dip_core.config import DIPConfig
from dip_core.pipeline.preprocess import load_image, add_synthetic_noise, generate_synthetic_inpaint_mask
from dip_core.pipeline.trainer import train_dip
from dip_core.pipeline.postprocess import apply_thermal_colormap
from dip_core.baselines import apply_mean_filter, apply_median_filter, apply_gaussian_filter
from dip_core.metrics import calculate_psnr, calculate_ssim

print(f"PyTorch version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 1. Create / Load Synthetic Thermal Image
Generate a synthetic thermal sensor pattern with high-temperature hot spots and gradient background.

In [ ]:
# Create synthetic thermal simulation (or load from file)
x = np.linspace(-3, 3, 256)
y = np.linspace(-3, 3, 256)
xx, yy = np.meshgrid(x, y)
thermal_sim = np.exp(-(xx**2 + yy**2) / 2.0) + 0.5 * np.exp(-((xx - 1.2)**2 + (yy - 1.0)**2) / 0.5)
thermal_sim = (thermal_sim - thermal_sim.min()) / (thermal_sim.max() - thermal_sim.min())

# Corrupt with Gaussian sensor noise
noisy_thermal = add_synthetic_noise(thermal_sim, noise_type="gaussian", std=0.15)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(apply_thermal_colormap(thermal_sim, 'ironbow'))
ax[0].set_title('Clean Thermal Reference')
ax[0].axis('off')

ax[1].imshow(apply_thermal_colormap(noisy_thermal, 'ironbow'))
ax[1].set_title(f'Noisy Input (PSNR: {calculate_psnr(noisy_thermal, thermal_sim):.2f} dB)')
ax[1].axis('off')
plt.tight_layout()
plt.show()

## 2. Train U-Net DIP Generator
Optimizes U-Net weights directly on the single noisy thermal frame without any training dataset.

In [ ]:
config = DIPConfig(
    iterations=600,
    learning_rate=0.01,
    noise_std=0.1,
    reg_noise_std=0.03,
    early_stopping=True,
)

result = train_dip(
    image_input=noisy_thermal,
    reference_input=thermal_sim,
    config=config,
    task="denoise"
)

restored = result["restored_image"]
metrics = result["metrics"]
print("Optimization Complete!")
print(f"Final PSNR: {metrics['psnr']} dB | SSIM: {metrics['ssim']}")

## 3. Compare Against Classical Filters

In [ ]:
mean_img = apply_mean_filter(noisy_thermal, ksize=5)
median_img = apply_median_filter(noisy_thermal, ksize=5)
gauss_img = apply_gaussian_filter(noisy_thermal, ksize=5)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(apply_thermal_colormap(noisy_thermal, 'ironbow'))
axes[0].set_title(f"Noisy ({calculate_psnr(noisy_thermal, thermal_sim):.2f} dB)")
axes[0].axis('off')

axes[1].imshow(apply_thermal_colormap(mean_img, 'ironbow'))
axes[1].set_title(f"Mean Filter ({calculate_psnr(mean_img, thermal_sim):.2f} dB)")
axes[1].axis('off')

axes[2].imshow(apply_thermal_colormap(gauss_img, 'ironbow'))
axes[2].set_title(f"Gaussian Filter ({calculate_psnr(gauss_img, thermal_sim):.2f} dB)")
axes[2].axis('off')

axes[3].imshow(apply_thermal_colormap(restored, 'ironbow'))
axes[3].set_title(f"U-Net DIP (Ours) ({metrics['psnr']} dB)")
axes[3].axis('off')
plt.tight_layout()
plt.show()

## 4. Plot Loss Convergence

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(result["loss_history"], label="Training Loss (MSE)", color="#f97316")
plt.yscale('log')
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("DIP Loss Convergence Curve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()